In [0]:
from pyspark.sql.functions import *
import builtins

In [0]:
df_bronze = (
    spark.read.table("retail_sales.bronze.sales_raw")
)
df_bronze.count()
#df_bronze.printSchema()
df_bronze.show()

###3 - Fase: corrugir o schema

In [0]:
order_date = to_date(col("ORDERDATE"), "M/d/yyyy H:mm")

df_silver = df_bronze.withColumns({
    'ORDERDATE': order_date,
    'YEAR_ID': year(order_date),
    'MONTH_ID': month(order_date)
})

#display(df_silver)

###4 - Fase: tratar nulo

In [0]:
def tratar_nullos(df: DataFrame) -> DataFrame:
    return df.dropna(subset=['ORDERNUMBER'])

df_silver = tratar_nullos(df_silver)
display(df_silver.limit(4))

### 5 - Fase: Remover Duplicados

In [0]:
df_silver = df_silver.dropDuplicates(["ORDERDATE", "ORDERLINENUMBER"])
#display(df_silver)

###6 - Fase: Padronizar dados

In [0]:
from pyspark.sql.types import DecimalType
def padronizar_dados_consistentes(df: DataFrame) -> DataFrame:
    df = df.withColumns({
        'STATUS': upper(col('STATUS')),
        'QTR_ID': quarter(col('ORDERDATE'))
    })
    df = df.withColumn(
    "STATUS",
    when(col("STATUS") == "SHIPPED", "ENVIADO")
    .when(col("STATUS") == "CANCELLED", "CANCELADO")
    .when(col("STATUS") == "IN PROCESS", "EM PROCESSAMENTO")
    .when(col("STATUS") == "RESOLVED", "RESOLVIDO")
    .when(col("STATUS") == "DISPUTED", "EM DISPUTA")
    .when(col("STATUS") == "ON HOLD", "EM ESPERA")
    .otherwise(col("STATUS"))
)
    return df
df_silver = padronizar_dados_consistentes(df_silver)
display()


###7 - Fase: coluna derivada (Notei algumas inconsistencias na coluna sales)

In [0]:
def colunas_derivadas(df: DataFrame) -> DataFrame:
    df = df.withColumns({
        "SALES_CALCULADO": col('QUANTITYORDERED') * col('PRICEEACH'),
        })
    return df

df_derivada = colunas_derivadas(df_silver)
#display(df_silver.limit(4))

###8 - Fase: Reorganizar o modelo(renomear as colunas , mudar a ordem)


In [0]:
def renomear_colunas(df: DataFrame) -> DataFrame:
    df_silver = df.select(
    col("ORDERNUMBER").alias("numero_pedido"),
    col("ORDERLINENUMBER").alias("numero_linha_pedido"),

    col("ORDERDATE").alias("data_pedido"),
    col("YEAR_ID").alias("ano"),
    col("QTR_ID").alias("trimestre"),
    col("MONTH_ID").alias("mes"),

    col("STATUS").alias("status_pedido"),

    col("QUANTITYORDERED").alias("quantidade_pedida"),
    col("PRICEEACH").alias("preco_unitario"),

    col("SALES").alias("vendas"),
    col("SALES_CALCULADO").alias("vendas_calculado")
    ).filter((col("vendas") > 0) & (col("vendas_calculado") > 0))
    return df_silver
df_renomear = renomear_colunas(df_derivada)
df_silver = df_renomear

tabela = "retail_sales.silver.sales_clean"

def salvar_ou_alterar(df: DataFrame, tabela: str) ->None:
    #Se a tabela nao existir criar
    if not spark.catalog.tableExists(tabela):
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    #se existir
    else:
        
        #verificar se o schema nao foi alterado
        from datetime import datetime
        
        if df.schema != spark.table(tabela).schema:
            log = {
                "pipeline": "sales_clean",
                "message": "schema alterado",
                "data": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "detalhes": "A API retornou ou a base de dados foi atualizada e retornou um schema diferente do esperado."
            }

            spark.createDataFrame([log]).write.mode("append").option("header", "true").option("mergeSchema", "true").saveAsTable("retail_sales.logs.pipelines_logs")
            print("Schema alterado.")
            raise Exception("Schema alterado.")

        #fazer o merge
        from delta.tables import DeltaTable
        tabela_silver = DeltaTable.forName(spark, tabela)

        tabela_silver.alias("origem") \
            .merge(df.alias("destino"), "origem.numero_pedido = destino.numero_pedido AND destino.numero_linha_pedido = origem.numero_linha_pedido") \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        log = {
                "pipeline": "sales_clean",
                "message": "dados atualizados",
                "data": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "detalhes": "Dados atualizados. com sucesso"
        }
        spark.createDataFrame([log]).write.mode("append").option("header", "true").option("mergeSchema", "true").saveAsTable("retail_sales.logs.pipelines_logs")
        print("Dados atualizados com sucesso.")

salvar_ou_alterar(df_silver, tabela)
display(df_silver.limit(3))


In [0]:
%sql
ANALYZE TABLE retail_sales.silver.sales_clean COMPUTE STATISTICS;